# Day 2 — Part 4: Submission Guidelines and Validation

This notebook acts as a **strict competition evaluator**. We extract all submission
requirements from the original guidelines, then check whether our current
agent and pipeline satisfy each one. For anything that fails, we provide a fix.

---

## Learning Objectives (from the original challenge)

By the end of this workshop, you should be able to:

1. Explain the difference between prompt-based, retrieval-augmented, and agentified AI systems
2. Describe the core components of an AI agent architecture (model, tools, control loop, memory, output schema)
3. Use the Claude API to build an AI application with tool calling and structured outputs
4. Design and implement a simple agent workflow (retrieve → tool execute → synthesize)
5. Model a real-world public-service problem as an ML and agent design challenge
6. Prepare and normalize municipal service data for retrieval and reasoning
7. Build a local retrieval layer using CSV and JSON data as grounding
8. Implement function-calling tools for jurisdiction routing
9. Generate grounded, structured answers with ownership, rationale, and next steps
10. Evaluate an agent using a benchmark set, task-specific metrics, and error analysis
11. Identify common failure modes (hallucinated authorities, ambiguity, overconfidence)
12. Compare baseline vs improved system variants
13. Apply reproducibility practices using DVC pipelines
14. Organize data preparation, evaluation, and reporting into a reusable ML pipeline
15. Reflect on agent architectures for public-sector problems

## Workshop Structure

| Phase | Duration | Activity |
|-------|----------|----------|
| 1 | 20 min | Instructor demo — agents, workflows, parameters |
| 2 | 65 min | Team notebook development (Day2_01 through Day2_03) |
| 3 | 5 min | Push final notebook to GitHub |
| 4 | During presentations | Instructor review and peer review |
| 5 | 1 min | Email .git link to instructor |

## Step 1: Submission Checklist with Pass/Fail Validation

We run each requirement as a programmatic check so the evaluation is objective.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT.resolve()}')

Project root: D:\Projects\AI_Agent_Workshop


In [2]:
# ── Automated submission validation ────────────────────────────────────────────
results = []

def check(requirement, condition, fix_if_fail=''):
    status = 'PASS' if condition else 'FAIL'
    results.append({
        'Requirement': requirement,
        'Status': status,
        'Fix Recommendation': fix_if_fail if not condition else 'N/A'
    })
    icon = '✓' if condition else '✗'
    print(f'  {icon} {status:4s}  {requirement}')

print('=== SUBMISSION VALIDATION REPORT ===')
print()

# ── File presence ──────────────────────────────────────────────────────────────
print('--- Files ---')

unified_notebook = ROOT / 'notebooks' / 'AI_Agent_Workflow.ipynb'
check('notebooks/AI_Agent_Workflow.ipynb exists',
      unified_notebook.exists(),
      'Create the unified notebook by running all Day2_0x notebooks together')

check('README.md exists', (ROOT / 'README.md').exists(),
      'Create README.md with project summary and team names')

check('requirements.txt exists', (ROOT / 'requirements.txt').exists(),
      'Run: pip freeze > requirements.txt (or create manually with anthropic, pandas, etc.)')

check('.gitignore exists', (ROOT / '.gitignore').exists(),
      'Create .gitignore to exclude .venv/, .env, artifacts/, __pycache__/')

check('src/ module package exists', (ROOT / 'src' / '__init__.py').exists(),
      'Create src/ directory with Python modules (schema, retrieval, tools, agent, evaluation, pipeline)')

# ── Data and artifacts ─────────────────────────────────────────────────────────
print()
print('--- Data and Artifacts ---')

check('data/service_catalog.csv exists',
      (ROOT / 'data' / 'service_catalog.csv').exists(),
      'Restore service_catalog.csv — it is the source of truth for the agent')

check('eval/service_eval_set.csv exists',
      (ROOT / 'eval' / 'service_eval_set.csv').exists(),
      'Restore evaluation set CSV file')

cleaned_json = ROOT / 'artifacts' / 'service_catalog.cleaned.json'
check('artifacts/service_catalog.cleaned.json exists (pipeline ran)',
      cleaned_json.exists(),
      'Run: python scripts/prepare_data.py')

metrics_json = ROOT / 'artifacts' / 'metrics.json'
check('artifacts/metrics.json exists (evaluation ran)',
      metrics_json.exists(),
      'Run: python scripts/run_agent_eval.py')

# ── Metrics quality ────────────────────────────────────────────────────────────
print()
print('--- Metrics Quality ---')

if metrics_json.exists():
    metrics = json.loads(metrics_json.read_text())
    check('metrics.json has all 5 required metrics',
          all(k in metrics for k in [
              'jurisdiction_accuracy', 'responsible_body_accuracy',
              'format_compliance_rate', 'avg_reasoning_quality', 'source_presence_rate'
          ]),
          'Update run_agent_eval.py to compute all 5 metrics (not just jurisdiction_accuracy)')
    
    check('jurisdiction_accuracy >= 0.5 (baseline acceptable)',
          metrics.get('jurisdiction_accuracy', 0) >= 0.5,
          'Improve retrieval or agent prompt — keyword baseline should hit >= 50%')
else:
    check('metrics.json has all 5 required metrics', False,
          'Run the full DVC pipeline first')
    check('jurisdiction_accuracy >= 0.5', False,
          'Run the full DVC pipeline first')

# ── DVC pipeline ───────────────────────────────────────────────────────────────
print()
print('--- DVC Pipeline ---')

check('dvc.yaml exists', (ROOT / 'dvc.yaml').exists(),
      'Create dvc.yaml with prepare_data, run_agent_eval, and report_metrics stages')

check('params.yaml exists', (ROOT / 'params.yaml').exists(),
      'Create params.yaml with retrieval.top_k, agent.model_name, etc.')

# ── Agent implementation ───────────────────────────────────────────────────────
print()
print('--- Agent Implementation ---')

check('src/agent.py exists (Claude API agent)',
      (ROOT / 'src' / 'agent.py').exists(),
      'Create src/agent.py with baseline_call(), grounded_call(), tool_agent_call()')

check('src/tools.py exists (tool declarations)',
      (ROOT / 'src' / 'tools.py').exists(),
      'Create src/tools.py with search_service_index, lookup_service_owner, suggest_next_steps')

check('src/evaluation.py exists (metrics)',
      (ROOT / 'src' / 'evaluation.py').exists(),
      'Create src/evaluation.py with evaluate_single() and compute_metrics()')

check('3-tier agent progression demonstrated',
      (ROOT / 'notebooks' / 'day2' / 'Day2_02_build_the_service_agent.ipynb').exists(),
      'Complete Day2_02 notebook showing Tier 1 (baseline), Tier 2 (RAG), Tier 3 (tool-calling)')

print()
print(f'Total checks: {len(results)}   PASS: {sum(1 for r in results if r["Status"]=="PASS")}   FAIL: {sum(1 for r in results if r["Status"]=="FAIL")}')

=== SUBMISSION VALIDATION REPORT ===

--- Files ---
  ✓ PASS  notebooks/AI_Agent_Workflow.ipynb exists
  ✓ PASS  README.md exists
  ✓ PASS  requirements.txt exists
  ✓ PASS  .gitignore exists
  ✓ PASS  src/ module package exists

--- Data and Artifacts ---
  ✓ PASS  data/service_catalog.csv exists
  ✓ PASS  eval/service_eval_set.csv exists
  ✓ PASS  artifacts/service_catalog.cleaned.json exists (pipeline ran)
  ✓ PASS  artifacts/metrics.json exists (evaluation ran)

--- Metrics Quality ---
  ✓ PASS  metrics.json has all 5 required metrics
  ✓ PASS  jurisdiction_accuracy >= 0.5 (baseline acceptable)

--- DVC Pipeline ---
  ✓ PASS  dvc.yaml exists
  ✓ PASS  params.yaml exists

--- Agent Implementation ---
  ✓ PASS  src/agent.py exists (Claude API agent)
  ✓ PASS  src/tools.py exists (tool declarations)
  ✓ PASS  src/evaluation.py exists (metrics)
  ✓ PASS  3-tier agent progression demonstrated

Total checks: 17   PASS: 17   FAIL: 0


In [3]:
# Display as a styled summary table
checklist_df = pd.DataFrame(results)

def color_status(val):
    return 'background-color: #d4edda' if val == 'PASS' else 'background-color: #f8d7da'

checklist_df.style.map(color_status, subset=['Status'])

,Requirement,Status,Fix Recommendation
0,notebooks/AI_Agent_Workflow.ipynb exists,PASS,N/A
1,README.md exists,PASS,N/A
2,requirements.txt exists,PASS,N/A
3,.gitignore exists,PASS,N/A
4,src/ module package exists,PASS,N/A
5,data/service_catalog.csv exists,PASS,N/A
6,eval/service_eval_set.csv exists,PASS,N/A
7,artifacts/service_catalog.cleaned.json exists (pipeline ran),PASS,N/A
8,artifacts/metrics.json exists (evaluation ran),PASS,N/A
9,metrics.json has all 5 required metrics,PASS,N/A


## Step 2: Quick Fix Commands

If any checks failed, run the corresponding fix command below.

In [4]:
failed = [r for r in results if r['Status'] == 'FAIL']

if not failed:
    print('All checks passed! Ready to submit.')
else:
    print(f'{len(failed)} issue(s) to fix:')
    for i, item in enumerate(failed, 1):
        print(f'\n{i}. {item["Requirement"]}')
        print(f'   Fix: {item["Fix Recommendation"]}')

All checks passed! Ready to submit.


In [5]:
# Run the full DVC pipeline to regenerate all artifacts
import subprocess
import os

print('Running DVC pipeline...')
result = subprocess.run(
    ['python', 'scripts/prepare_data.py'],
    cwd=str(ROOT), capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

result2 = subprocess.run(
    ['python', 'scripts/run_agent_eval.py'],
    cwd=str(ROOT), capture_output=True, text=True
)
print(result2.stdout)
if result2.returncode != 0:
    print('ERROR:', result2.stderr)

Running DVC pipeline...




## Step 3: GitHub Submission Checklist

Before pushing, verify each item manually:

| Item | Done |
|------|-------|
| All notebooks in `notebooks/` with executed outputs saved | Yes |
| `AI_Agent_Workflow.ipynb` is the primary submission artifact | Yes |
| `README.md` includes project summary, approach, and team names | Yes |
| `.gitignore` excludes `.venv/`, `.env`, `artifacts/*.json` | Yes |
| `requirements.txt` is up to date | Yes |
| No API keys or secrets are committed to Git | Yes |
| Repo is public and named `AI_Agent_Workshop` | Yes |
| At least one meaningful commit beyond the initial push | Yes |

---

## What We Built — Summary

| Component | Description |
|-----------|-------------|
| **Schema** | Strict TypedDict + JSON schema for all agent responses |
| **Tier 1 — Baseline** | Claude called directly with no grounding |
| **Tier 2 — RAG** | Keyword retrieval + catalog context injected into prompt |
| **Tier 3 — Tool Agent** | Claude calls 3 tools (search, lookup, next_steps) in an agentic loop |
| **Evaluation** | 5 metrics: jurisdiction accuracy, body accuracy, format compliance, reasoning quality, source presence |
| **Pipeline** | 3-stage DVC pipeline: prepare_data → run_agent_eval → report_metrics |
| **Modules** | 6 src/ modules: schema, retrieval, tools, agent, evaluation, pipeline |